# Benchmark InternVL3.5-8B — 355 keyframe AIC

Model "a quan" trong bang khao sat, truoc bi loai vi "vuot 7B cua de bai". Ly do do
da do: PDF chinh thuc cua BTC khong noi gi ve tham so, tran that tren T4 la ~11 ty.

8,53 ty tham so, kien truc InternVLChatModel — TRUNG voi Vintern nen dung chung
adapter, khong can code moi. Uoc ~7 GB VRAM (Vintern-3B do duoc he so 0,765 GB/ty).

Ghim transformers >=4.52.1: InternVL3.5 dung Qwen3 lam phan ngon ngu, ban <4.50 cua
Vintern bao loi "cannot import name Qwen3Config". Hai ho InternVL khong chung moc.

Thu tu cell: chay -> nen/luu -> kiem.

In [ ]:
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')
assert torch.cuda.is_available(), 'Chua bat GPU'


In [ ]:
# InternVL3.5 dung Qwen3 lam phan ngon ngu -> can transformers >=4.52.1.
# KHAC Vintern (<4.50): hai ho InternVL nay khong dung chung moc thu vien.
!pip install -q "transformers>=4.52.1,<5" accelerate bitsandbytes timm einops sentencepiece
import transformers
print('transformers:', transformers.__version__)
assert transformers.__version__ >= '4.52.1', f'pip keo nham ban {transformers.__version__}'


In [ ]:
import subprocess, sys
from pathlib import Path

REPO = 'https://github.com/lolizabrett-byte/Multimodal-Agentic-Retrieval-Engine.git'
NHANH = 'research/vlm-prompting'
DICH = Path('/kaggle/working/repo')

# Kaggle giu /kaggle/working giua cac version, nen "clone neu chua co" se dung
# code cu cua lan chay truoc. Da mat mot luot vi vay: adapter moi khong duoc goi,
# loi cu lap lai y het. Xoa roi clone lai moi lan.
import shutil
if DICH.exists():
    shutil.rmtree(DICH)
subprocess.run(['git', 'clone', '--depth', '1', '-b', NHANH, REPO, str(DICH)], check=True)

PKG = DICH / 'system1' / 'research' / 'vlm_prompting'
assert PKG.exists(), f'Khong thay code tai {PKG}'
sys.path.insert(0, str(PKG))

# Xoa module da import o lan chay truoc, neu khong Python dung ban cu trong bo nho.
for ten in list(sys.modules):
    if ten.startswith(('vlm', 'benchmark_runner', 'checkpoint_utils', 'quality')):
        del sys.modules[ten]

hash_code = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                           cwd=DICH, capture_output=True, text=True).stdout.strip()
print('Code tai:', PKG)
print('Commit:', hash_code)
assert hash_code, 'Khong doc duoc commit hash -- clone that bai'


In [ ]:
ANH_DIR = next(Path('/kaggle/input').glob('**/images'), None)
print('Thu muc anh:', ANH_DIR)
so_anh = len(list(ANH_DIR.glob('*.jpg')))
print('So anh:', so_anh)
assert so_anh >= 100, f'Chi co {so_anh} anh, de bai can >= 100'

In [ ]:
lenh = [
    sys.executable, 'scripts/benchmark_runner.py',
    '--mode', 'mass',
    '--models', 'internvl35-8b',
    '--backend', 'transformers',
    '--strict', '--restart',
    '--frames-dir', str(ANH_DIR),
    '--out-dir', '/kaggle/working/ket_qua',
]
print('Chay:', ' '.join(lenh))
kq = subprocess.run(lenh, cwd=str(PKG))
print('Ma thoat:', kq.returncode)
# Khong assert -- ket qua phai duoc nen truoc khi bat cu thu gi nem loi.

In [ ]:
import shutil
ZIP = shutil.make_archive('/kaggle/working/benchmark-internvl35', 'zip', '/kaggle/working/ket_qua')
print('Da nen:', ZIP)

In [ ]:
import json
ra = Path('/kaggle/working/ket_qua')
for f in sorted(ra.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(ra)}  {f.stat().st_size:,} bytes')
ck = ra / 'checkpoint_internvl35-8b.json'
assert ck.exists(), 'Khong thay checkpoint -- model khong nap duoc, doc log o tren'
d = json.loads(ck.read_text(encoding='utf-8'))
xong = d.get('da_xong') or {}
ok = sum(1 for v in xong.values() if v.get('thanh_cong'))
raw = sum(1 for v in xong.values() if v.get('raw_text'))
print(f'{len(xong)} anh | JSON hop le {ok} ({ok/max(len(xong),1):.1%}) | co raw_text {raw}')
for v in list(xong.values())[:2]:
    if v.get('raw_text'):
        print('vi du output hong:', v['raw_text'][:300])
bang = ra / 'vlm_comparison_results.json'
if bang.exists():
    b = json.loads(bang.read_text(encoding='utf-8'))
    for mk, v in (b.get('ket_qua') or {}).items():
        print(f'{mk}: vram={v.get("vram_dinh_gb")} GB | lat={v.get("latency_trung_binh")} s')